# L04 — Population Stability Index (PSI) & Model Monitoring
**Credit Risk Modeling | Lending Club Dataset**

---
## Learning Objectives
1. Understand **why models degrade** over time and why monitoring is critical
2. Learn the **PSI formula** and decision thresholds
3. Compute PSI for **every input variable** to detect feature-level drift
4. Compute PSI for **credit scores** — the most critical monitoring signal
5. Interpret results and decide: *Is our model still valid?*
6. Build a **complete monitoring dashboard** with action recommendations

---
## The core question this notebook answers:
> *"We trained our PD model on 2007–2014 data. One year later, do 2015 borrowers look similar enough for the model to still work?"*

---
## Prerequisites
- L01, L02, L03 complete
- `train_preprocessed.parquet` (reference population — 2007–2014)
- `test_preprocessed.parquet` (monitoring population — 2015)
- `scorecard.csv` from L02


## 1. Setup & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score
import warnings, json, os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'stable':'#2ecc71','monitor':'#f39c12','alert':'#e74c3c','blue':'#3498db'}
os.makedirs('../data/reports', exist_ok=True)

# Reference = training population (what the model was built on)
# Actual    = monitoring population (new borrowers 2015)
reference = pd.read_parquet('../data/processed/train_preprocessed.parquet')
actual    = pd.read_parquet('../data/processed/test_preprocessed.parquet')

with open('../data/processed/dummy_cols.json') as f:
    DUMMY_COLS = json.load(f)

print(f"Reference (train 2007-2014): {len(reference):,} loans")
print(f"Actual    (OOT  2015):       {len(actual):,} loans")
print(f"Reference default rate: {1-reference['good_bad'].mean():.2%}")
print(f"Actual    default rate: {1-actual['good_bad'].mean():.2%}")

## 2. What is Population Stability Index (PSI)?

### The Problem

Credit risk models assume the future will resemble the past. But borrower populations change because of:
- **Economic cycles** — recessions bring riskier applicants
- **Policy changes** — new underwriting rules change who applies
- **Competitor actions** — rivals attract certain borrower segments

If the 2015 population is very different from 2007–2014, the model's predictions will be unreliable — even if it was excellent when validated.

### The PSI Formula

$$PSI = \sum_{i=1}^{n} \left( A_i - E_i \right) \times \ln\left(\frac{A_i}{E_i}\right)$$

Where:
- $A_i$ = **Actual** % in bucket $i$ (new/monitoring population)
- $E_i$ = **Expected** % in bucket $i$ (training/reference population)
- $n$ = number of buckets (10 for continuous, or number of categories for discrete)

### Decision Thresholds

| PSI | Traffic Light | Interpretation | Action |
|-----|--------------|----------------|--------|
| **< 0.10** | 🟢 STABLE | No significant change | Continue as-is |
| **0.10 – 0.25** | 🟡 MONITOR | Moderate shift | Investigate; consider recalibration |
| **> 0.25** | 🔴 ALERT | Major shift | Model likely needs to be rebuilt |

### Why PSI = 0 means no change

If $A_i = E_i$ for every bucket, then $(A_i - E_i) = 0$ for every term → PSI = 0.
As distributions diverge, each term grows (both the difference and the log ratio increase in magnitude), so PSI grows.

### Score PSI has a stricter interpretation

For the **credit score** specifically:
- PSI > 0.10 → investigate immediately
- PSI > 0.19 → seriously consider rebuilding
- PSI > 0.25 → model must be redeveloped

This is stricter because score drift directly affects cut-off decisions and provisioning calculations.


In [2]:
# ── PSI helper functions ─────────────────────────────────────────────────

def psi_status(psi_value):
    if   psi_value < 0.10: return ('STABLE',  COLORS['stable'])
    elif psi_value < 0.25: return ('MONITOR', COLORS['monitor'])
    else:                  return ('ALERT',   COLORS['alert'])

def psi_discrete(ref_series, act_series):
    all_cats = sorted(set(ref_series.dropna().unique()) | set(act_series.dropna().unique()))
    rows = []
    for cat in all_cats:
        e = max((ref_series == cat).mean(), 1e-6)
        a = max((act_series == cat).mean(), 1e-6)
        rows.append({'Category': str(cat),
                     'Expected_%': e*100, 'Actual_%': a*100,
                     'PSI_contrib': (a-e)*np.log(a/e)})
    df = pd.DataFrame(rows)
    df['PSI'] = df['PSI_contrib'].sum()
    return df

def psi_continuous(ref_series, act_series, n_buckets=10):
    ref = ref_series.copy()
    act = act_series.copy()
    ref_miss = (ref == -999)
    act_miss = (act == -999)
    ref_act  = ref[~ref_miss].dropna()
    act_act  = act[~act_miss].dropna()

    # Breakpoints from REFERENCE only
    pct = np.linspace(0, 100, n_buckets+1)
    bp  = np.percentile(ref_act, pct)
    bp[0] = -np.inf; bp[-1] = np.inf

    rows = []
    if ref_miss.any() or act_miss.any():
        e = max(ref_miss.mean(), 1e-6)
        a = max(act_miss.mean(), 1e-6)
        rows.append({'Bucket':'Missing','Expected_%':e*100,'Actual_%':a*100,
                     'PSI_contrib':(a-e)*np.log(a/e)})

    for i in range(n_buckets):
        lo, hi = bp[i], bp[i+1]
        e = max(((ref_act>=lo)&(ref_act<hi)).mean(), 1e-6)
        a = max(((act_act>=lo)&(act_act<hi)).mean(), 1e-6)
        rows.append({'Bucket':f'[{lo:.2f},{hi:.2f})',
                     'Expected_%':e*100,'Actual_%':a*100,
                     'PSI_contrib':(a-e)*np.log(a/e)})

    df = pd.DataFrame(rows)
    df['PSI'] = df['PSI_contrib'].sum()
    return df

print("PSI functions defined. Thresholds: <0.10 Stable | 0.10-0.25 Monitor | >0.25 Alert")

PSI functions defined. Thresholds: <0.10 Stable | 0.10-0.25 Monitor | >0.25 Alert


## 3. PSI for Discrete Variables

We compare the **% of borrowers in each category** between the reference (training 2007–2014) and actual (2015) populations.

**What to watch:**
- A big shift in `grade` distribution means the model's calibrated WoE coefficients may no longer apply
- A shift in `purpose` or `home_ownership` may indicate a strategic shift in the lending product mix
- Shifts driven by bank strategy (e.g., new marketing targeting homeowners) are less concerning than genuine economic shifts


In [ ]:
discrete_vars = ['grade','home_ownership','verification_status',
                 'purpose','initial_list_status']

psi_results = {}
fig, axes = plt.subplots(2, 3, figsize=(16,9))
axes = axes.flatten()

for idx, var in enumerate(discrete_vars):
    if var not in reference.columns:
        axes[idx].set_visible(False)
        continue
    df_p = psi_discrete(reference[var], actual[var])
    psi_val = df_p['PSI'].iloc[0]
    psi_results[var] = psi_val
    status, color = psi_status(psi_val)

    cats = df_p['Category'].values
    x = np.arange(len(cats))
    w = 0.38
    axes[idx].bar(x-w/2, df_p['Expected_%'], w, label='Ref (2007-14)',
                  color=COLORS['blue'], alpha=0.75)
    axes[idx].bar(x+w/2, df_p['Actual_%'],   w, label='Act (2015)',
                  color=color, alpha=0.75)
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(cats, rotation=45, ha='right', fontsize=7)
    axes[idx].set_title(f'{var}\nPSI={psi_val:.4f} [{status}]',
                        fontsize=9, color=color if status!='STABLE' else 'black')
    axes[idx].set_ylabel('% loans')
    if idx == 0: axes[idx].legend(fontsize=7)

axes[-1].set_visible(False)
plt.suptitle('PSI — Discrete Variables', fontsize=12)
plt.tight_layout()
plt.savefig('../data/reports/L04_psi_discrete_variables.png', dpi=150, bbox_inches='tight')
plt.show()

print("Discrete Variable PSI:")
for var, psi in sorted(psi_results.items(), key=lambda x:-x[1]):
    s, _ = psi_status(psi)
    flag = '🔴' if s=='ALERT' else ('🟡' if s=='MONITOR' else '🟢')
    print(f"  {flag} {var:<35} PSI={psi:.4f}  [{s}]")

## 4. PSI for Continuous Variables

For continuous variables we use **10 equal-frequency buckets** defined on the **reference population only**.

> **Important:** Bucket boundaries must always come from the reference (training) set. Using the actual population's quantiles would hide the very shift we want to detect — because the buckets would move with the distribution.

**Interpreting continuous PSI:**

If `int_rate` PSI is high, it means the interest rate distribution has shifted. This could mean:
1. The bank is pricing differently (internal policy)
2. Economic conditions changed (interest rate environment)
3. The mix of loan grades has shifted (which affects rates indirectly)

In all cases, the model's WoE bins for `int_rate` may no longer accurately capture default risk.


In [4]:
continuous_vars = {
    'int_rate':                    'Interest Rate',
    'annual_inc':                  'Annual Income',
    'dti':                         'Debt-to-Income',
    'mths_since_issue_d':          'Months Since Issue',
    'mths_since_earliest_cr_line': 'Credit History (months)',
    'emp_length_int':              'Employment Length',
}

psi_cont = {}
for var, label in continuous_vars.items():
    if var in reference.columns:
        df_p = psi_continuous(reference[var], actual[var])
        psi_cont[var] = {'label': label, 'psi': df_p['PSI'].iloc[0], 'df': df_p}

print("Continuous Variable PSI:")
print(f"{'Variable':<42} {'PSI':>8}  Status")
print("-"*60)
for var, info in sorted(psi_cont.items(), key=lambda x:-x[1]['psi']):
    s, _ = psi_status(info['psi'])
    flag = '🔴' if s=='ALERT' else ('🟡' if s=='MONITOR' else '🟢')
    print(f"  {flag} {var:<40} {info['psi']:>8.4f}  [{s}]")

Continuous Variable PSI:
Variable                                        PSI  Status
------------------------------------------------------------
  🔴 mths_since_issue_d                        12.7624  [ALERT]
  🟡 mths_since_earliest_cr_line                0.1656  [MONITOR]
  🟢 int_rate                                   0.0836  [STABLE]
  🟢 emp_length_int                             0.0111  [STABLE]
  🟢 annual_inc                                 0.0086  [STABLE]
  🟢 dti                                        0.0078  [STABLE]


In [ ]:
# Detailed chart for top 4 continuous variables
top4 = sorted(psi_cont.items(), key=lambda x:-x[1]['psi'])[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, (var, info) in zip(axes.flatten(), top4):
    df_p = info['df']
    psi_val = info['psi']
    status, color = psi_status(psi_val)

    x = np.arange(len(df_p))
    w = 0.38
    ax.bar(x-w/2, df_p['Expected_%'], w, label='Ref (2007-14)',
           color=COLORS['blue'], alpha=0.75)
    ax.bar(x+w/2, df_p['Actual_%'],   w, label='Act (2015)',
           color=color, alpha=0.75)
    ax.set_xticks(x)
    ax.set_xticklabels([str(b)[:10] for b in df_p['Bucket']],
                       rotation=45, ha='right', fontsize=7)
    ax.set_title(f'{info["label"]}\nPSI={psi_val:.4f} [{status}]',
                 color=color if status!='STABLE' else 'black')
    ax.set_ylabel('% of loans')
    if ax == axes[0][0]: ax.legend(fontsize=8)

plt.suptitle('PSI — Key Continuous Variables (Top 4 by PSI)', fontsize=12)
plt.tight_layout()
plt.savefig('../data/reports/L04_psi_continuous_variables.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Credit Score PSI — The Most Important Signal

The credit score is the **aggregated output** of the model. Score PSI captures the net effect of all individual variable shifts.

Even if each individual variable PSI is only 0.05 (green), the compounding effect can push score PSI above 0.10–0.25 (yellow/red).

### What score PSI tells you in plain English:
- **PSI < 0.10** → "2015 applicants look like 2007–2014 applicants. The scorecard still applies."
- **PSI 0.10–0.25** → "2015 applicants score differently on average. Our cut-offs may approve a different mix than intended."
- **PSI > 0.25** → "The 2015 population is fundamentally different. The model is scoring applicants it was not designed for."


In [6]:
# Load scorecard and compute scores
try:
    scorecard = pd.read_csv('../data/processed/scorecard.csv')
    score_map = dict(zip(scorecard['Feature'], scorecard['Score']))
    FACTOR = 20 / np.log(2)
    OFFSET = 600

    def compute_scores(df):
        s = pd.Series(0.0, index=df.index)
        for feat, sc in score_map.items():
            if feat in df.columns:
                s += df[feat] * sc
        # Add distributed intercept offset
        s += OFFSET
        return s.clip(300, 850).round().astype(int)

    ref_scores = compute_scores(reference)
    act_scores = compute_scores(actual)
    print(f"Reference: mean={ref_scores.mean():.1f}  std={ref_scores.std():.1f}")
    print(f"Actual:    mean={act_scores.mean():.1f}  std={act_scores.std():.1f}")

except FileNotFoundError:
    print("scorecard.csv not found. Using simulated scores for demo.")
    np.random.seed(42)
    ref_scores = pd.Series(np.random.normal(615, 80, len(reference)).clip(300,850).astype(int))
    act_scores = pd.Series(np.random.normal(630, 85, len(actual)).clip(300,850).astype(int))

Reference: mean=680.0  std=27.4
Actual:    mean=688.5  std=25.2


In [7]:
# Score PSI
df_score_psi = psi_continuous(ref_scores, act_scores, n_buckets=10)
score_psi = df_score_psi['PSI'].iloc[0]
status, color = psi_status(score_psi)

print(f"\n{'='*50}")
print(f"  CREDIT SCORE PSI = {score_psi:.4f}  [{status}]")
print(f"{'='*50}")
print()
print(df_score_psi[['Bucket','Expected_%','Actual_%','PSI_contrib']].round(4).to_string(index=False))


  CREDIT SCORE PSI = 0.1093  [MONITOR]

         Bucket  Expected_%  Actual_%  PSI_contrib
  [-inf,644.00)      9.9797    4.0032       0.0546
[644.00,656.00)      9.5171    7.1641       0.0067
[656.00,665.00)      9.5147    8.0670       0.0024
[665.00,673.00)      9.7602    8.5099       0.0017
[673.00,681.00)     10.4675    9.4917       0.0010
[681.00,688.00)      9.4038    9.2387       0.0000
[688.00,696.00)     10.6098   11.1345       0.0003
[696.00,704.00)      9.8144   11.5150       0.0027
[704.00,715.00)     10.6451   14.6050       0.0125
   [715.00,inf)     10.2878   16.2709       0.0274


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlapping distributions
axes[0].hist(ref_scores, bins=40, alpha=0.55, color=COLORS['blue'],
             label=f'Reference (2007-14) n={len(ref_scores):,}', density=True)
axes[0].hist(act_scores, bins=40, alpha=0.55, color=color,
             label=f'Actual (2015) n={len(act_scores):,}', density=True)
axes[0].set(xlabel='Credit Score', ylabel='Density',
            title=f'Score Distribution Comparison\nPSI = {score_psi:.4f}  [{status}]')
axes[0].legend()

# Cumulative distributions (shows shift direction)
ref_sorted = np.sort(ref_scores)
act_sorted = np.sort(act_scores)
axes[1].plot(ref_sorted, np.linspace(0,1,len(ref_sorted)),
             color=COLORS['blue'], lw=2.5, label='Reference (2007-14)')
axes[1].plot(act_sorted, np.linspace(0,1,len(act_sorted)),
             color=color, lw=2.5, label='Actual (2015)')
axes[1].set(xlabel='Credit Score', ylabel='Cumulative %',
            title='Cumulative Score Distribution\n(Gap = shift in borrower quality)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/reports/L04_score_psi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretation
print("Interpretation of score distribution shift:")
if ref_scores.mean() < act_scores.mean():
    print(f"  2015 borrowers score HIGHER on average (+{act_scores.mean()-ref_scores.mean():.1f} pts)")
    print("  → 2015 applicants appear less risky / better quality on paper")
    print("  → Could reflect: economic improvement, tighter initial underwriting, or model drift")
else:
    print(f"  2015 borrowers score LOWER on average ({act_scores.mean()-ref_scores.mean():.1f} pts)")
    print("  → 2015 applicants appear riskier on paper")
    print("  → Could reflect: economic deterioration or riskier applicant mix")

## 6. Full PSI Summary Dashboard

In [ ]:
# Build master summary
all_psi = {}
for var, psi in psi_results.items():
    all_psi[var] = {'psi': psi, 'type': 'Discrete'}
for var, info in psi_cont.items():
    all_psi[var] = {'psi': info['psi'], 'type': 'Continuous'}
all_psi['credit_score'] = {'psi': score_psi, 'type': 'Score (KEY)'}

psi_df = pd.DataFrame([
    {'Variable': v, 'PSI': d['psi'], 'Type': d['type'],
     'Status': psi_status(d['psi'])[0]}
    for v, d in all_psi.items()
]).sort_values('PSI', ascending=False)

# Color-coded horizontal bar chart
fig, ax = plt.subplots(figsize=(11, max(6, len(psi_df)*0.45)))
bar_colors = [psi_status(p)[1] for p in psi_df['PSI']]
bars = ax.barh(psi_df['Variable'], psi_df['PSI'], color=bar_colors, alpha=0.85)
ax.axvline(0.10, color='gold', lw=2, ls='--', label='Monitor (0.10)')
ax.axvline(0.25, color='red',  lw=2, ls='--', label='Alert (0.25)')
ax.set(xlabel='PSI', title='Population Stability Index — All Variables\n(🟢 Stable | 🟡 Monitor | 🔴 Alert)')
ax.legend(loc='lower right')
for bar, val in zip(bars, psi_df['PSI']):
    ax.text(val+0.003, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../data/reports/L04_psi_master_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== MASTER PSI REPORT ===")
print(f"{'Variable':<35} {'Type':<15} {'PSI':>8}  Status")
print("-"*68)
for _, row in psi_df.iterrows():
    flag = '🔴' if row['Status']=='ALERT' else ('🟡' if row['Status']=='MONITOR' else '🟢')
    print(f"  {flag} {row['Variable']:<33} {row['Type']:<15} {row['PSI']:>8.4f}  {row['Status']}")

## 7. Interpretation & Recommended Actions

### How to diagnose PSI findings

**Step 1 — Look at score PSI first.**
Score PSI is the headline number. It tells you whether the model's output distribution has shifted overall.

**Step 2 — Drill into which variables are driving the shift.**
High individual variable PSI tells you *where* the shift originates.

**Step 3 — Distinguish strategic vs. genuine drift.**
- If `grade` PSI is high because the bank changed its marketing → less concerning
- If `dti` PSI is high because the economy has changed → more concerning

**Step 4 — Validate model performance on new data.**
PSI is a leading indicator. Confirm with actual Gini/KS computed on the monitoring population.

### Decision Framework

```
Score PSI < 0.10
  → STABLE. No action. Continue quarterly monitoring.

0.10 ≤ Score PSI < 0.19
  → MODERATE. Investigate root cause.
  → Run Gini/KS on 2015 data.
  → If Gini drop < 5pp: increase monitoring to monthly.
  → If Gini drop 5-10pp: consider intercept recalibration.

0.19 ≤ Score PSI < 0.25
  → SIGNIFICANT. High-priority investigation.
  → Consider retraining on more recent data.
  → Update coarse class boundaries for high-PSI variables.

Score PSI > 0.25
  → CRITICAL. Begin model redevelopment process.
  → New model should use more recent training data.
  → Consider XGBoost/LightGBM + SHAP for next version.
```

### What "recalibration" means vs. "rebuild"

| | Recalibration | Rebuild |
|--|--|--|
| **When** | Gini still good, scores shifted | Gini degraded significantly |
| **What changes** | Intercept, score scaling, cut-offs | All coefficients, features, WoE bins |
| **Effort** | 1-2 weeks | 2-3 months |
| **Regulatory filing** | Minor update | Full model documentation |


In [10]:
# ── Performance check on monitoring population ────────────────────────────
print("=== MODEL PERFORMANCE MONITORING TEMPLATE ===")
print()
print("Run this code after computing pd_pred_test from L02:")
print()
print("  from sklearn.metrics import roc_auc_score, brier_score_loss")
print("  from scipy.stats import ks_2samp")
print()
print("  auc_ref  = roc_auc_score(y_train, 1 - pd_pred_train)")
print("  auc_oot  = roc_auc_score(y_test,  1 - pd_pred_test)")
print("  gini_ref = 2 * auc_ref - 1")
print("  gini_oot = 2 * auc_oot - 1")
print("  ks_oot   = ks_2samp(pd_pred_test[y_test==1],")
print("                       pd_pred_test[y_test==0]).statistic")
print()
gini_thresholds = pd.DataFrame({
    'Metric':    ['Gini (OOT)','KS (OOT)','AUC (OOT)','Score PSI','Variable PSI'],
    'Green':     ['>0.40',     '>0.25',    '>0.65',    '<0.10',    '<0.10'],
    'Yellow':    ['0.35-0.40','0.20-0.25','0.60-0.65','0.10-0.25','0.10-0.25'],
    'Red':       ['<0.35',     '<0.20',    '<0.60',    '>0.25',    '>0.25'],
    'Action if Red': ['Rebuild','Rebuild','Rebuild','Rebuild/Recalib','Re-encode bins'],
})
print(gini_thresholds.to_string(index=False))

=== MODEL PERFORMANCE MONITORING TEMPLATE ===

Run this code after computing pd_pred_test from L02:

  from sklearn.metrics import roc_auc_score, brier_score_loss
  from scipy.stats import ks_2samp

  auc_ref  = roc_auc_score(y_train, 1 - pd_pred_train)
  auc_oot  = roc_auc_score(y_test,  1 - pd_pred_test)
  gini_ref = 2 * auc_ref - 1
  gini_oot = 2 * auc_oot - 1
  ks_oot   = ks_2samp(pd_pred_test[y_test==1],
                       pd_pred_test[y_test==0]).statistic

      Metric Green    Yellow   Red   Action if Red
  Gini (OOT) >0.40 0.35-0.40 <0.35         Rebuild
    KS (OOT) >0.25 0.20-0.25 <0.20         Rebuild
   AUC (OOT) >0.65 0.60-0.65 <0.60         Rebuild
   Score PSI <0.10 0.10-0.25 >0.25 Rebuild/Recalib
Variable PSI <0.10 0.10-0.25 >0.25  Re-encode bins


In [11]:
# ── Save results ─────────────────────────────────────────────────────────
os.makedirs('../data/processed', exist_ok=True)
psi_df['monitoring_date'] = '2015'
psi_df['reference_period'] = '2007-2014'
psi_df.to_csv('../data/processed/psi_results.csv', index=False)

n_stable  = (psi_df['Status']=='STABLE').sum()
n_monitor = (psi_df['Status']=='MONITOR').sum()
n_alert   = (psi_df['Status']=='ALERT').sum()

print("PSI results saved to ../data/processed/psi_results.csv")
print()
print("=== FINAL MONITORING VERDICT ===")
print(f"  🟢 STABLE:  {n_stable}  variables")
print(f"  🟡 MONITOR: {n_monitor} variables")
print(f"  🔴 ALERT:   {n_alert}  variables")
print()

sc_row = psi_df[psi_df['Variable']=='credit_score']
if len(sc_row):
    sc_psi = sc_row['PSI'].iloc[0]
    sc_status = sc_row['Status'].iloc[0]
    print(f"Credit Score PSI = {sc_psi:.4f} → {sc_status}")
    print()
    if sc_status == 'STABLE':
        print("RECOMMENDATION: Model is stable.")
        print("  → Continue standard quarterly monitoring.")
        print("  → No model changes needed.")
    elif sc_status == 'MONITOR':
        print("RECOMMENDATION: Score distribution has shifted noticeably.")
        print("  → Investigate which variables are driving the shift.")
        print("  → Validate Gini/KS on 2015 data.")
        print("  → Consider intercept recalibration if Gini is still acceptable.")
        print("  → Increase monitoring frequency to monthly.")
    else:
        print("RECOMMENDATION: MAJOR population shift detected.")
        print("  → Begin model redevelopment process immediately.")
        print("  → Retrain on 2013-2015 data (more recent window).")
        print("  → Next version: consider XGBoost + SHAP for higher robustness.")

PSI results saved to ../data/processed/psi_results.csv

=== FINAL MONITORING VERDICT ===
  🟢 STABLE:  8  variables
  🟡 MONITOR: 2 variables
  🔴 ALERT:   2  variables

Credit Score PSI = 0.1093 → MONITOR

RECOMMENDATION: Score distribution has shifted noticeably.
  → Investigate which variables are driving the shift.
  → Validate Gini/KS on 2015 data.
  → Consider intercept recalibration if Gini is still acceptable.
  → Increase monitoring frequency to monthly.


## V3 Improvements — Characteristic Stability Index (CSI) & Champion/Challenger

> **What changed:** The original L04 computed PSI for the score and a few variables. V3 adds CSI (PSI at the feature level) to diagnose *what's causing* score drift, and adds a champion/challenger framework to govern model promotion decisions.

### V3-8: Characteristic Stability Index (CSI) per Input Variable
**Why needed:** Score PSI = 0.1093 (MONITOR) tells us the distribution has shifted, but not *which variables are responsible*. CSI applies the identical PSI formula to each input predictor individually.

- Same thresholds as PSI: < 0.10 Stable → 0.10–0.25 Monitor → > 0.25 Alert
- Differentiates **strategic drift** (bank changed its marketing) vs **economic drift** (DTI rising economy-wide)
- High CSI on a variable → re-encode WoE bins for that variable in the next model update cycle
- This satisfies the SR 11-7 requirement for "input monitoring" distinct from "output monitoring"

### V3-11: Champion/Challenger Testing Framework
**Why needed:** When a new model version (challenger) is ready, how do you decide whether to promote it? The standard industry protocol requires **two simultaneous tests:**

1. **Practical significance:** Gini (challenger) − Gini (champion) ≥ 0.02
2. **Statistical significance:** Mann-Whitney U test p-value < 0.05

Both must pass. Gini delta alone is insufficient because small datasets can show apparent improvements by chance. The Mann-Whitney test is preferred over the DeLong test because it is non-parametric and works on the raw score distributions without assuming normality.

**Production deployment pattern:**
1. Shadow mode: challenger runs in parallel, makes no actual decisions
2. After 6+ months, compare outcomes on the same population
3. Run champion/challenger test on the parallel outcomes
4. Promote only when both conditions hold

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# V3-8: CHARACTERISTIC STABILITY INDEX (CSI) per Input Variable
# ────────────────────────────────────────────────────────────────────────────
# CSI = PSI formula applied to each individual input variable.
# Identifies WHICH specific predictors are driving score distribution shift.

def characteristic_stability_index(ref_series, live_series, n_bins=10):
    """PSI applied to an individual predictor variable (called CSI)."""
    ref_clean  = ref_series.replace(-1, np.nan).dropna()
    live_clean = live_series.replace(-1, np.nan).dropna()
    try:
        bins = pd.qcut(ref_clean, q=n_bins, duplicates='drop', retbins=True)[1]
        bins[0] = -np.inf;  bins[-1] = np.inf
        ref_cnt  = pd.cut(ref_clean,  bins=bins).value_counts(normalize=True).sort_index().clip(lower=1e-6)
        live_cnt = pd.cut(live_clean, bins=bins).value_counts(normalize=True).sort_index().clip(lower=1e-6)
        live_cnt = live_cnt.reindex(ref_cnt.index).fillna(1e-6)
        return abs(float(((live_cnt - ref_cnt) * np.log(live_cnt / ref_cnt)).sum()))
    except Exception:
        return np.nan

csi_vars = {
    'int_rate':                    'Interest Rate',
    'annual_inc':                  'Annual Income',
    'dti':                         'Debt-to-Income Ratio',
    'mths_since_issue_d':          'Months Since Issue',
    'mths_since_earliest_cr_line': 'Credit History Length',
    'emp_length_int':              'Employment Length',
}

csi_rows = []
for var, label in csi_vars.items():
    if var in reference.columns and var in actual.columns:
        csi_val = characteristic_stability_index(reference[var], actual[var])
        status_label, _ = psi_status(csi_val)
        csi_rows.append({'Variable': var, 'Label': label, 'CSI': csi_val, 'Status': status_label})

csi_df = pd.DataFrame(csi_rows).sort_values('CSI', ascending=False)

print("=== V3-8: Characteristic Stability Index (CSI) ===")
print(f"{'Predictor':<42} {'CSI':>8}  Status")
print("-"*60)
for _, row in csi_df.iterrows():
    flag = '🔴' if row['Status']=='ALERT' else ('🟡' if row['Status']=='MONITOR' else '🟢')
    print(f"  {flag} {row['Label']:<40} {row['CSI']:>8.4f}  [{row['Status']}]")

fig, ax = plt.subplots(figsize=(10, max(4, len(csi_df)*0.55)))
bar_colors = [psi_status(c)[1] for c in csi_df['CSI']]
ax.barh(csi_df['Label'], csi_df['CSI'], color=bar_colors, alpha=0.85)
ax.axvline(0.10, color='gold', lw=2, ls='--', label='Monitor (0.10)')
ax.axvline(0.25, color='red',  lw=2, ls='--', label='Alert (0.25)')
ax.set(xlabel='CSI', title='Characteristic Stability Index per Input Variable\n(CSI = PSI applied to each predictor; same thresholds)')
ax.legend(loc='lower right')
for i, (_, row) in enumerate(csi_df.iterrows()):
    ax.text(row['CSI']+0.003, i, f'{row["CSI"]:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../data/reports/L04_characteristic_stability_index.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCSI Interpretation:")
print("  CSI tells you WHICH predictors are causing score distribution shift.")
print("  High CSI → re-encode WoE bins for that variable in the next model update.")
print("  Pair CSI with PSI: if score PSI is high but all CSI are low → check feature interactions.")
print("  Action: variables with CSI > 0.25 require re-binning before next monitoring cycle.")

# ────────────────────────────────────────────────────────────────────────────
# V3-11: CHAMPION / CHALLENGER TESTING FRAMEWORK
# ────────────────────────────────────────────────────────────────────────────

print("\n=== V3-11: Champion/Challenger Testing Framework ===")

def champion_challenger_test(y_true, champion_proba, challenger_proba,
                              alpha=0.05, min_gini_delta=0.02):
    """
    Promote challenger only if BOTH conditions hold:
    1. Gini delta >= min_gini_delta  (practical significance)
    2. Mann-Whitney U p < alpha      (statistical significance)
    """
    auc_champ  = roc_auc_score(y_true, champion_proba)
    auc_chal   = roc_auc_score(y_true, challenger_proba)
    gini_delta = 2 * (auc_chal - auc_champ)
    stat, p_value = mannwhitneyu(
        challenger_proba[y_true == 0],
        challenger_proba[y_true == 1],
        alternative='greater'
    )
    promote = (gini_delta >= min_gini_delta) and (p_value < alpha)
    return {
        'gini_champion':   round(2*auc_champ - 1, 4),
        'gini_challenger': round(2*auc_chal  - 1, 4),
        'gini_delta':      round(gini_delta,       4),
        'mw_p_value':      round(p_value,           6),
        'verdict':         'PROMOTE CHALLENGER' if promote else 'KEEP CHAMPION'
    }

# Load score arrays computed in this session
try:
    scorecard_csv = pd.read_csv('../data/processed/scorecard.csv')
    feat_map = dict(zip(scorecard_csv['Feature'], scorecard_csv['Score']))
    OFFSET_SC, FACTOR_SC = 600, 20 / np.log(2)

    def score_loans(df):
        s = sum(df[f] * v for f, v in feat_map.items() if f in df.columns)
        return pd.Series(s + OFFSET_SC, index=df.index).clip(300, 850) / 850.0

    champ_proba = score_loans(reference).values
    y_ref       = reference['good_bad'].values

    # Simulate challenger: add small Gaussian noise (replace with real challenger in production)
    np.random.seed(42)
    chal_proba = np.clip(champ_proba + np.random.normal(0, 0.02, len(champ_proba)), 0, 1)

    cc = champion_challenger_test(y_ref, champ_proba, chal_proba)

    print(f"  Champion  Gini: {cc['gini_champion']:.4f}")
    print(f"  Challenger Gini:{cc['gini_challenger']:.4f}")
    print(f"  Gini Delta:     {cc['gini_delta']:+.4f}  (threshold ≥ +0.02)")
    print(f"  MW p-value:     {cc['mw_p_value']:.6f}  (α = 0.05)")
    print(f"\n  >>> Verdict: {cc['verdict']} <<<")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].hist(champ_proba[y_ref==1], bins=50, alpha=0.5, color='#2ecc71',
                 label='Champion — Good', density=True)
    axes[0].hist(champ_proba[y_ref==0], bins=50, alpha=0.5, color='#e74c3c',
                 label='Champion — Bad',  density=True)
    axes[0].set(xlabel='P(Good)', ylabel='Density',
                title=f'Champion | Gini={cc["gini_champion"]:.4f}')
    axes[0].legend()

    axes[1].hist(chal_proba[y_ref==1], bins=50, alpha=0.5, color='#3498db',
                 label='Challenger — Good', density=True)
    axes[1].hist(chal_proba[y_ref==0], bins=50, alpha=0.5, color='#e67e22',
                 label='Challenger — Bad',  density=True)
    axes[1].set(xlabel='P(Good)', ylabel='Density',
                title=f'Challenger | Gini={cc["gini_challenger"]:.4f} (Δ={cc["gini_delta"]:+.4f})')
    axes[1].legend()

    plt.suptitle(f'Champion vs Challenger Test — {cc["verdict"]}', fontsize=11)
    plt.tight_layout()
    plt.savefig('../data/reports/L04_champion_challenger.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nProduction Use:")
    print("  1. Deploy challenger in shadow mode: same decisions, no actual impact.")
    print("  2. Collect 6+ months of parallel outcomes.")
    print("  3. Run this test on the parallel outcomes dataset.")
    print("  4. Promote challenger only when BOTH conditions hold.")

except Exception as e:
    print(f"  Champion/Challenger demo: {e}")
    print("  In production, replace champ_proba/chal_proba with actual model output arrays.")

## Summary & Key Takeaways

### PSI is the primary early-warning system for credit models

| PSI Level | Monitored Object | Tells You |
|-----------|-----------------|-----------|
| **Input variable PSI** | Each feature (grade, income, rate...) | Which inputs are drifting and why |
| **Credit score PSI** | Model output distribution | Whether the overall model is still applicable |
| **Gini/KS on new data** | Model discrimination on OOT | Whether discrimination has actually degraded |

### Complete 4-Notebook Learning Journey

| Notebook | Core Deliverable |
|----------|-----------------|
| **L01** | Cleaned data · WoE bins · Dummy variables · OOT split |
| **L02** | PD logistic model · Scorecard (300-850) · 10 risk classes · ROI credit policy |
| **L03** | LGD 2-stage model · EAD model · EL = PD × LGD × EAD · Portfolio analysis |
| **L04** | PSI per variable · Score PSI · Monitoring verdict · Action framework |

### From Notebooks to Production

The notebooks have taught you the **what** and **why** of every step. The `Credit_Risk_Implementation_Plan.md` describes how to operationalize these steps at scale using:

```
Lending Club CSV (millions of rows)
    → PySpark cleaning & feature jobs
    → PostgreSQL (raw / staging / features / models / risk schemas)
    → Apache Airflow DAG orchestration
    → MLflow model registry (Staging → Production gate)
    → FastAPI real-time scoring endpoint
    → Prometheus + Grafana monitoring dashboards
    → Daily PSI DAG with auto-alerts at PSI > 0.25
```

**Every function in these notebooks maps 1:1 to a production module in the implementation plan.**
